<a href="https://colab.research.google.com/github/ddyyaanaaa-ship-it/Danva-EDA/blob/main/P02_Praktikum_Asysyifa_Ainaya_Zahra.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Praktikum Pertemuan 2 — pandas di Google Colab
**Mata Kuliah:** Exploratory Data Analysis (KMTI21133) — Pertemuan 2
**Sub-CPMK2:** Mahasiswa mampu menggunakan Python (pandas) di Google Colab/Jupyter dan Git/GitHub untuk memuat, menelusuri, dan menyimpan data serta kode analisis.

**Cara kerja:** sel bertanda `# DEMO` diisi bersama dosen; sel bertanda `# TODO` Anda kerjakan sendiri. Setiap TODO punya *expected output* agar Anda bisa memeriksa jawaban sendiri. Simpan notebook ini ke repositori tim di akhir sesi (Bagian E).

## Bagian 0 — Persiapan
Jalankan sel di bawah. Jika file `penjualan_umkm_2024.csv` belum ada di sesi Colab, sel akan meminta Anda mengunggahnya (ikon folder di kiri → Upload).

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 120)
print('pandas', pd.__version__, '| numpy', np.__version__)

pandas 2.2.3 | numpy 2.1.3


In [ ]:
import os
FILE = 'penjualan_umkm_2024.csv'

if not os.path.exists(FILE):
    try:
        from google.colab import files
        files.upload()          # pilih penjualan_umkm_2024.csv
    except ImportError:
        raise FileNotFoundError('/content/drive/MyDrive/Semester 3/EDA2026/Pertemuan02/Data/penjualan_umkm_2024.csv')
print('Siap:', os.path.exists(FILE))

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


---
## Bagian A — Memuat dan memeriksa data (Latihan 1)
**Tujuan:** sebelum menganalisis apa pun, jawab empat pertanyaan: *berapa besar data ini, apa saja kolomnya, apa tipe datanya, dan seberapa bersih?*

In [ ]:
# DEMO A1 — memuat data
df = pd.read_csv('penjualan_umkm_2024.csv')
df.head()

In [ ]:
# DEMO A2 — ukuran dan struktur
print('Baris, kolom:', df.shape)
df.info()

In [ ]:
# DEMO A3 — ringkasan numerik
df.describe()

> **Catatan dosen:** `describe()` secara default hanya meringkas kolom numerik. Coba `df.describe(include='object')` untuk kolom teks.

In [ ]:
# TODO A4
# a. Tampilkan 5 baris TERAKHIR dari df
df.tail()
print(df.tail())

# b. Tampilkan daftar nama kolom beserta tipe datanya
df.dtypes
print(df.dtypes)

# Expected: 1212 baris x 11 kolom; kolom 'tanggal' bertipe object (belum datetime)


In [ ]:
# TODO A5 — seberapa bersih data ini?
# a. Hitung jumlah nilai kosong (missing) per kolom
df.isna().sum()
print(df.isna().sum())

# b. Hitung jumlah baris duplikat
df.duplicated().sum()
print(df.duplicated().sum())

# c. Hitung berapa nilai unik pada kolom 'kota'
df['kota'].unique()
print(df['kota'].unique())

# Expected: rating_pelanggan & metode_bayar memiliki missing; ada 12 baris duplikat;
#           kolom 'kota' punya LEBIH dari 5 nilai unik -- mengapa? (diskusikan)


**Diskusi (2 menit):** mengapa jumlah nilai unik `kota` lebih banyak daripada jumlah kota sebenarnya? Masalah seperti ini akan kita tangani secara sistematis pada pertemuan 4–5.

---
## Bagian B — Seleksi dan filter (Latihan 2, berpasangan)
**Tujuan:** mengambil bagian data yang relevan dengan pertanyaan analisis.

In [ ]:
# DEMO B1 — memilih kolom
df['total_bayar'].head()          # satu kolom -> Series
df[['kota','kategori','total_bayar']].head()   # beberapa kolom -> DataFrame

In [ ]:
# DEMO B2 — loc (berbasis label) vs iloc (berbasis posisi)
print(df.loc[0, 'produk'])
print(df.iloc[0, 4])
df.loc[0:4, ['produk','jumlah','total_bayar']]

In [ ]:
# DEMO B3 — filter dengan kondisi (boolean mask)
mask = df['total_bayar'] > 1_000_000
print('Transaksi di atas 1 juta:', mask.sum())
df[mask].head()

In [ ]:
# DEMO B4 — menggabungkan kondisi
hi = df[(df['kategori'] == 'Makanan') & (df['diskon_persen'] >= 10)]
print(hi.shape)
df[df['metode_bayar'].isin(['QRIS','E-Wallet'])].shape

In [ ]:
# TODO B5
# a. Ambil semua transaksi dengan rating_pelanggan < 3
rating_rendah = df[df['rating_pelanggan'] < 3]
print(rating_rendah)

# b. Berapa banyak transaksi tersebut dan berapa rata-rata total_bayar-nya?
jml_rating_rendah = rating_rendah.shape[0]
rata_rata_total_bayar = rating_rendah['total_bayar'].mean()

print("Banyak transaksi (rating < 3):", jml_rating_rendah)
print("Rata-rata total_bayar (rating < 3):", rata_rata_total_bayar)

# c. Ambil transaksi kategori 'Kerajinan' ATAU 'Fashion' dengan jumlah >= 10
rating_rendah_fashion_kerajinan = (df['kategori'].isin(['Kerajinan', 'Fashion'])) & (df['jumlah'] >= 10)
transaksi_khusus = df[rating_rendah_fashion_kerajinan]

print("Transaksi Kerajinan/Fashion dengan jumlah >= 10")
print("Transaksi khusus",transaksi_khusus)


In [ ]:
# TODO B6 — pertanyaan bisnis
# 'Transaksi tunai bernilai besar (> 500.000) di kota mana saja?'
transaksi_tunai = df[(df['metode_bayar'] == 'Tunai') & (df['total_bayar'] > 500000)]
kota_transaksi_tunai = transaksi_tunai['kota'].unique()

print("Transaksi tunai bernilai besar (> 500.000):")
print(transaksi_tunai)
print("Kota-kota yang melakukan transaksi tunai bernilai besar:")
print(kota_transaksi_tunai)

# Tampilkan kolom kota, produk, jumlah, total_bayar, diurutkan dari total_bayar terbesar.
hasil = df[df['metode_bayar'] == 'Tunai'][['kota', 'produk', 'jumlah', 'total_bayar']].sort_values(by='total_bayar', ascending=False)

print("Hasil:")
print(hasil)

# Petunjuk: gunakan .sort_values(..., ascending=False) dan .head(10)


---
## Bagian C — Kolom turunan dan pengurutan

In [ ]:
# DEMO C1 — membuat kolom baru
df['harga_bruto'] = df['jumlah'] * df['harga_satuan']
df['hemat'] = df['harga_bruto'] - df['total_bayar']
df[['harga_bruto','total_bayar','hemat']].head()

In [ ]:
# DEMO C2 — kolom tanggal
df['tanggal'] = pd.to_datetime(df['tanggal'])
df['bulan'] = df['tanggal'].dt.month
df['nama_hari'] = df['tanggal'].dt.day_name()
df[['tanggal','bulan','nama_hari']].head()

In [ ]:
# TODO C3
# a. Buat kolom 'kelas_nilai' berisi 'Besar' jika total_bayar > 500000, selain itu 'Kecil'
#    Petunjuk: np.where(kondisi, 'Besar', 'Kecil')
df['kelas_nilai'] = np.where(df['total_bayar'] > 500000, 'Besar', 'Kecil')
print(df[['total_bayar', 'kelas_nilai']].head())

# b. Hitung berapa banyak transaksi pada tiap kelas (petunjuk: value_counts())
jml_transaksi_per_kelas_nilai = df['kelas_nilai'].value_counts()
print("jumlah transaksi:")
print(jml_transaksi_per_kelas_nilai)

# c. Tampilkan 5 transaksi dengan penghematan (kolom 'hemat') terbesar
transaksi_hemat_terbesar = df.sort_values(by='hemat', ascending=False).head(5)
print("lima transaksi hemat terbesar:")
print(transaksi_hemat_terbesar)

---
## Bagian D — Agregasi dengan groupby
Pola yang akan Anda pakai sepanjang semester: **split → apply → combine**.

In [ ]:
# DEMO D1 — agregasi sederhana
df.groupby('kategori')['total_bayar'].mean().round(0)

In [ ]:
# DEMO D2 — beberapa agregasi sekaligus
ringkas = (df.groupby('kategori')
             .agg(jumlah_transaksi=('id_transaksi','count'),
                  total_penjualan=('total_bayar','sum'),
                  rata_rata_nilai=('total_bayar','mean'),
                  rata_rata_rating=('rating_pelanggan','mean'))
             .round(2)
             .sort_values('total_penjualan', ascending=False))
ringkas

In [ ]:
# DEMO D3 — dua kunci pengelompokan
df.groupby(['kategori','metode_bayar'])['total_bayar'].sum().unstack(fill_value=0)

In [ ]:
# TODO D4
# a. Total penjualan per kota, diurutkan dari yang terbesar
total_penjualan_kota = (
    df.groupby('kota')['total_bayar']
      .sum()
      .sort_values(ascending=False)
)

print("a. Total penjualan per kota:")
print(total_penjualan_kota)

# b. Rata-rata rating per kategori. Kategori mana yang ratingnya paling rendah?
rata_rating_kategori = (
    df.groupby('kategori')['rating_pelanggan']
      .mean()
      .sort_values()
)

print("\nb. Rata-rata rating per kategori:")
print(rata_rating_kategori)

print(
    "\nKategori dengan rating paling rendah:",
    rata_rating_kategori.idxmin()
)

# c. Penjualan per bulan. Bulan mana yang tertinggi?
penjualan_bulan = (
    df.groupby('bulan')['total_bayar']
      .sum()
      .sort_values(ascending=False)
)

print("\nc. Penjualan per bulan:")
print(penjualan_bulan)

print(
    "\nBulan dengan penjualan tertinggi:",
    penjualan_bulan.idxmax()
)

In [ ]:
# TODO D5 — satu temuan
# Tulis SATU kalimat temuan dari hasil di atas dalam sel Markdown berikutnya.
# Format: 'X memiliki ... sebesar ..., lebih tinggi/rendah dibandingkan ...'
# Ingat: ini temuan DESKRIPTIF, belum boleh diklaim sebagai sebab-akibat.


 _Metode pembayaran Tunai memiliki rata-rata total transaksi sebesar Rp505.958, lebih tinggi dibandingkan E-Wallet yang sebesar Rp496.473._

---
## Bagian E — Menyimpan hasil dan mengunggah ke repositori tim

In [ ]:
# DEMO E1 — menyimpan hasil agregasi
ringkas.to_csv('ringkasan_kategori.csv')
print('Tersimpan.')
# Di Colab: files.download('ringkasan_kategori.csv') untuk mengunduh

### Alur kerja repositori tim (dilakukan bersama di kelas)
```
nama-tim-eda/
├── README.md              <- tujuan proyek, anggota, cara menjalankan
├── requirements.txt
├── data/
│   ├── raw/               <- data asli, TIDAK PERNAH diubah
│   └── processed/         <- hasil olahan
├── notebooks/             <- 01_quality.ipynb, 02_cleaning.ipynb, ...
├── knime/
└── reports/
```
**Dua cara menyimpan notebook ke GitHub:**
1. Colab → *File → Save a copy in GitHub* → pilih repo tim → tulis pesan commit.
2. Melalui terminal/Git lokal:
```bash
git clone https://github.com/<organisasi-tim>/<nama-repo>.git
cd <nama-repo>
git checkout -b latihan-pert2
git add notebooks/P02_praktikum.ipynb
git commit -m 'Latihan pert 2: eksplorasi awal data penjualan UMKM'
git push origin latihan-pert2
```
**Aturan pesan commit di kelas ini:** kalimat perintah, maksimal ±60 karakter, menjelaskan *apa* dan *mengapa*.
Contoh baik: `Tambah agregasi penjualan per kota`. Contoh buruk: `update`, `fix`, `asdf`.

In [ ]:
# TODO E2 — checklist sebelum commit
# 1. Restart runtime lalu Run all -> pastikan tidak ada error
# 2. Hapus sel percobaan yang tidak terpakai
# 3. Pastikan nama file mengikuti pola: P02_praktikum_<nama>.ipynb
# 4. Commit & push, lalu salin tautan commit Anda ke LMS


---
## Bagian F — Checkpoint proyek (dikerjakan bersama tim, di luar kelas)
Terapkan Bagian A pada **dataset calon proyek tim Anda**, lalu jawab di README repositori:
1. Berapa baris dan kolom? Apa satuan tiap barisnya (satu baris = satu apa)?
2. Kolom mana yang numerik, kategorikal, dan tanggal?
3. Kolom mana yang bermasalah (kosong, tipe salah, nilai tidak konsisten)?
4. Apakah data ini cukup untuk menjawab pertanyaan analisis tim? Jika belum, apa yang kurang?

Hasil poin 1–4 menjadi bahan **M1 (proposal proyek)** yang dikumpulkan pada pertemuan 3.